# SNR pretraining eval on Google Colab (A100)

Mirrors the cluster eval pipeline (`scripts/evaluate.sbatch` +
`scripts/_run_per_task.sh`) but on a single A100 instead of 4×GPU.

**What it does:**
- HF model only (no Megatron — the local `Megatron-LM` source isn't here).
- One task at a time, `--log_samples --write_out`, results land in
  `<LOGS_ROOT>/<WANDB_ENTITY>/<WANDB_PROJECT>/<NAME>/harness/eval_<ts>_colab/per_task/<task>/`
  — same layout the cluster's `_run_per_task.sh` uses, so the cluster's
  idempotency check (`scripts/_eval_status.py`) recognises these results
  if you sync them back.
- Skips already-finished tasks for the same `NAME` based on what's on disk.
- Times each task and the total run.
- Merges per-task dirs and uploads to W&B with the same naming convention
  (`update_wandb_alignment.py`).

**Differences vs cluster:**
- `tensor_parallel_size=1`, `data_parallel_size=1` (single A100). vLLM
  uses `gpu_memory_utilization=0.85`, `max_model_len=4096`.
- Tasks run with `vllm` backend (same as cluster's HF path).
- No splits — Colab is one node.

**Persistence:** results live under `/content/eval_logs/` by default
(lost when the runtime ends). Mount Google Drive in the optional cell to
keep them across sessions, or `rsync` the dir to the cluster after each run.

Runtime needed: **A100 (40 GB ok for ≤7B)**. Set via
*Runtime → Change runtime type → A100*.

## 1. Install deps

Mirrors `scripts/evaluate.sbatch`'s `INSTALL_CMD`. Pinning to the same
versions the cluster uses so eval numbers are comparable.

In [ ]:
%%bash
set -e
pip install --quiet --no-cache-dir sentencepiece tiktoken protobuf
pip install --quiet --no-cache-dir --upgrade 'git+https://github.com/swiss-ai/lm-evaluation-harness.git'
pip install --quiet --no-cache-dir --upgrade \
    'huggingface-hub>=1.3.0,<2.0' \
    'transformers==5.1.0' \
    'accelerate>=1.1.0' \
    'antlr4-python3-runtime==4.11' \
    datasketch segtok 'tokenizers==0.22.2' mistral-common \
    sympy spacy math_verify latex2sympy2_extended wonderwords \
    nltk immutabledict langdetect hf-transfer subset2evaluate \
    'alpaca-eval>=0.6.2' 'peft>=0.18.0'
pip install --quiet --no-cache-dir 'vllm>=0.6.0' wandb
python -c 'import sentencepiece, tiktoken, vllm; print("sentencepiece", sentencepiece.__version__, "tiktoken", tiktoken.__version__, "vllm", vllm.__version__)'

## 2. GPU + auth

Confirm the A100 is visible, then load tokens. Add `HF_TOKEN` and
`WANDB_API_KEY` as Colab secrets (left sidebar → key icon → `+ Add new
secret`, name them exactly that, toggle *Notebook access*) — the cell
below pulls them from secrets or, as a fallback, from
`getpass()` prompts.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('Loaded HF_TOKEN + WANDB_API_KEY from Colab secrets.')
except Exception:
    if not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
    if not os.environ.get('WANDB_API_KEY'):
        os.environ['WANDB_API_KEY'] = getpass.getpass('WANDB_API_KEY: ')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # faster downloads
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

## 3. (Optional) mount Google Drive for persistence

Skip this cell to use ephemeral `/content/eval_logs/`. Mount Drive to
keep results across sessions; `LOGS_ROOT` will be set in the next cell.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # then in the next cell set: LOGS_ROOT = '/content/drive/MyDrive/swissai_eval_logs'

## 4. Configure the run

Edit `MODEL` / `REVISION` for the checkpoint to evaluate. `NAME` and the
directory layout match what the cluster produces — see
`scripts/evaluate.sbatch:160` (`NAME=$2`, `RUN_ROOT=$LOGS_ROOT/...`).

In [ ]:
# === Edit these ===
MODEL    = 'HuggingFaceTB/SmolLM3-3B-checkpoints'
REVISION = 'stage3-step-4720000'

# === Defaults match scripts/launch_evaluations.sh / hf_base_runner.sh ===
WANDB_ENTITY  = 'mariagrandury-epflnlp'
WANDB_PROJECT = 'snr-experiments'
LOGS_ROOT     = '/content/eval_logs'   # change to '/content/drive/MyDrive/...' if mounted

# Task list — same union the cluster uses for snr-pretraining-full.
TASKS_FILE_URL = 'https://raw.githubusercontent.com/mariagrandury/swissai-evals-post-train/main/configs/signal_to_ratio/tasks_pretraining_full.txt'

# Eval args — match COMMON_EVAL_ARGS in evaluate.sbatch
BATCH_SIZE      = 'auto:20'
MAX_BATCH_SIZE  = 32
MAX_NEW_TOKENS  = 2048
MAX_MODEL_LEN   = 4096
GPU_MEM_UTIL    = 0.85         # single A100, leave a bit of headroom
TP              = 1            # single GPU
TRUST_REMOTE    = True
ENABLE_THINKING = False
ADD_BOS         = False        # Megatron path uses BOS=true; for HF leave default

# Derived
import os, re
_repo_basename = MODEL.split('/')[-1]
NAME = f'{_repo_basename}-{REVISION}'
RUN_ROOT     = os.path.join(LOGS_ROOT, WANDB_ENTITY, WANDB_PROJECT, NAME)
HARNESS_DIR  = os.path.join(RUN_ROOT, 'harness')
os.makedirs(HARNESS_DIR, exist_ok=True)

from datetime import datetime
EVAL_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
HARNESS_EVAL_DIR = os.path.join(HARNESS_DIR, f'eval_{EVAL_TS}_colab')
PER_TASK_DIR     = os.path.join(HARNESS_EVAL_DIR, 'per_task')
os.makedirs(PER_TASK_DIR, exist_ok=True)

# Pull task list
import urllib.request
with urllib.request.urlopen(TASKS_FILE_URL) as r:
    raw = r.read().decode()
ALL_TASKS = [l.strip() for l in raw.splitlines()
             if l.strip() and not l.strip().startswith('#')]
print(f'NAME             = {NAME}')
print(f'HARNESS_EVAL_DIR = {HARNESS_EVAL_DIR}')
print(f'Tasks total      = {len(ALL_TASKS)}')
print('First 5:', ALL_TASKS[:5])

## 5. Idempotency — figure out what's still missing

Same logic as `scripts/_eval_status.py`: a task is *done* if there's a
non-empty `eval_*/per_task/<task>/` (killed run) or some
`eval_*/results_*.json` lists it under `.results` (clean run).

In [ ]:
import json, glob

def completed_tasks(harness_dir: str) -> set:
    done = set()
    if not os.path.isdir(harness_dir):
        return done
    for d in glob.glob(os.path.join(harness_dir, 'eval_*/per_task/*')):
        if os.path.isdir(d) and os.listdir(d):
            done.add(os.path.basename(d))
    for f in glob.glob(os.path.join(harness_dir, 'eval_*/results_*.json')):
        try:
            data = json.loads(open(f).read())
            done.update((data.get('results') or {}).keys())
        except Exception:
            pass
    return done

DONE = completed_tasks(HARNESS_DIR)
REMAINING = [t for t in ALL_TASKS if t not in DONE]
print(f'Already done: {len(DONE)}')
print(f'Remaining   : {len(REMAINING)}')
if REMAINING:
    print('First 10 remaining:', REMAINING[:10])
else:
    print('Nothing to do — all tasks already have results for this NAME.')

## 6. Run lm_eval per task

One subprocess per task, same pattern as `scripts/_run_per_task.sh`.
Failures in one task are logged to `failed_tasks.log` and the loop
continues — survivor metrics still reach W&B.

In [ ]:
import subprocess, time, shlex

MODEL_ARGS_PARTS = [
    'dtype=bfloat16',
    f'pretrained={MODEL}',
    f'tokenizer={MODEL}',
    f'revision={REVISION}',
    f'tokenizer_revision={REVISION}',  # SmolLM3-3B-checkpoints' main is empty — required
    f'data_parallel_size=1',
    f'tensor_parallel_size={TP}',
    f'gpu_memory_utilization={GPU_MEM_UTIL}',
    f'enable_thinking={str(ENABLE_THINKING)}',
    f'max_model_len={MAX_MODEL_LEN}',
]
if ADD_BOS:
    MODEL_ARGS_PARTS.insert(1, 'add_bos_token=True')
MODEL_ARGS = ','.join(MODEL_ARGS_PARTS)

BASE_ARGS = [
    'lm_eval',
    '--model', 'vllm',
    f'--model_args={MODEL_ARGS}',
    '--trust_remote_code' if TRUST_REMOTE else '',
    '--batch_size', str(BATCH_SIZE),
    '--max_batch_size', str(MAX_BATCH_SIZE),
    '--log_samples',
    '--write_out',
    '--confirm_run_unsafe_code',
    '--gen_kwargs', f'max_gen_toks={MAX_NEW_TOKENS}',
]
BASE_ARGS = [a for a in BASE_ARGS if a]

FAILED_LOG  = os.path.join(HARNESS_EVAL_DIR, 'failed_tasks.log')
SKIPPED_LOG = os.path.join(HARNESS_EVAL_DIR, 'skipped_tasks.log')
open(FAILED_LOG,  'w').close()
open(SKIPPED_LOG, 'w').close()

success_dirs = []
skipped_count = len(ALL_TASKS) - len(REMAINING)
if skipped_count:
    with open(SKIPPED_LOG, 'w') as f:
        f.write('\n'.join(t for t in ALL_TASKS if t in DONE) + '\n')
    print(f'(Skipping {skipped_count} task(s) with existing results.)')

wall_start = time.time()
for i, task in enumerate(REMAINING, 1):
    out = os.path.join(PER_TASK_DIR, task)
    cmd = BASE_ARGS + ['--tasks', task, '--output_path', out]
    pretty = ' '.join(shlex.quote(c) for c in cmd)
    print(f'\n=== [{i}/{len(REMAINING)}] Running task: {task} ===')
    print(pretty)
    t0 = time.time()
    rc = subprocess.run(cmd).returncode
    dt = time.time() - t0
    if rc == 0:
        success_dirs.append(out)
        print(f'=== OK: {task}  ({dt:.1f}s) ===')
    else:
        with open(FAILED_LOG, 'a') as f:
            f.write(task + '\n')
        print(f'=== FAILED (rc={rc}): {task}  — logged and continuing  ({dt:.1f}s) ===')

wall = time.time() - wall_start
EVAL_DURATION = int(wall)
print(f'\nTotal wall time: {wall:.1f}s ({wall/60:.1f} min)')
print(f'Successes: {len(success_dirs)}/{len(REMAINING)}')

## 7. Merge per-task dirs and upload to W&B

Reuses the cluster's own scripts (`scripts/alignment/merge_split_results.py`
and `update_wandb_alignment.py`) for full naming parity.

Cloning the repo gives us those modules; we don't need anything else from it.

In [ ]:
%%bash
if [ ! -d /content/swissai-evals-post-train ]; then
    git clone --depth 1 https://github.com/mariagrandury/swissai-evals-post-train.git /content/swissai-evals-post-train
fi

In [ ]:
import sys, subprocess
REPO = '/content/swissai-evals-post-train'
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# 1. merge per-task dirs into HARNESS_EVAL_DIR
if success_dirs:
    merge_cmd = ['python', '-m', 'scripts.alignment.merge_split_results',
                 '--split_dirs', *success_dirs,
                 '--output_dir', HARNESS_EVAL_DIR]
    print(' '.join(merge_cmd))
    subprocess.run(merge_cmd, cwd=REPO, check=True)
    # drop per_task scratch (matches _run_per_task.sh post-merge cleanup)
    import shutil
    shutil.rmtree(PER_TASK_DIR, ignore_errors=True)
else:
    print('No successful tasks — skipping merge.')

In [ ]:
# 2. upload to W&B with the same NAME
if success_dirs:
    upload_cmd = ['python', '-m', 'scripts.alignment.update_wandb_alignment',
                  '--entity', WANDB_ENTITY,
                  '--project', WANDB_PROJECT,
                  '--logs_root', HARNESS_EVAL_DIR,
                  '--name', NAME,
                  '--main_metrics', '',
                  '--eval_duration', str(EVAL_DURATION)]
    print(' '.join(upload_cmd))
    subprocess.run(upload_cmd, cwd=REPO, check=False)
else:
    print('No successful tasks — skipping W&B upload.')

## 8. (Optional) sync results back to the cluster

If you want the cluster's idempotency check to see Colab's results so
the next `launch_evaluations.sh` skips them, copy the eval dir over.
Run from the cluster login node, not Colab.

```bash
# On the cluster login node — example (paths from this notebook's NAME / EVAL_TS):
rsync -av <colab_or_drive_path>/eval_logs/<entity>/<project>/<NAME>/harness/eval_<ts>_colab/ \
    /iopsstor/scratch/cscs/mariagrandury/data-mix-small/Megatron-LM/logs/eval_logs/<entity>/<project>/<NAME>/harness/eval_<ts>_colab/
```